In [ ]:
import pandas as pd
import numpy as np

<h2>Load Dataset</h2>

In [ ]:
df = pd.read_csv("car_data.csv")

In [ ]:
df.head()

<h2>Feature Engineering</h2>

In [ ]:
df['Car_Age'] = 2026 - df['Year']
df['Brand'] = df['Car_Name'].str.split().str[0].str.lower()
df['Brand']

In [ ]:
#Calculate Goodwill feature

#Step 1 - Make a new feature called Brand
brand_stats = df.groupby('Brand').agg(
    Avg_Present_Price = ('Present_Price', 'mean'),  
    Num_Cars          = ('Car_Name', 'count'),        
    Avg_Age           = ('Car_Age', 'mean'),          
    Avg_Kms           = ('Driven_kms', 'mean')       
).round(2)

#Step 2 - calculate necessary items
brand_stats['Score_Price']      = brand_stats['Avg_Present_Price']
brand_stats['Score_Popularity'] = brand_stats['Num_Cars']       
brand_stats['Score_Usage']      = brand_stats['Avg_Kms']

#Step 3 - Finally get Goodwill feature 
brand_stats['Goodwill_Score'] = (
    0.5 * brand_stats['Score_Price'] +
    0.3 * brand_stats['Score_Popularity'] +
    0.2 * brand_stats['Score_Usage']
).round(4)

df['Brand_Goodwill_Score'] = df['Brand'].map(brand_stats['Goodwill_Score'])

print(brand_stats[['Goodwill_Score']].sort_values('Goodwill_Score', ascending=False))

In [ ]:
#Calculate Mileage feature from Driven_kms and Car_Age columns

df['Mileage_per_year'] = df['Driven_kms']/df['Car_Age'].replace(0,np.nan)
df['Mileage_per_year'] = df['Mileage_per_year'].fillna(df['Driven_kms'])
df['Mileage_per_year'] = pd.to_numeric(df['Mileage_per_year'].round(2))

In [ ]:
#Calculate Horsepower feature by taking Selling_Price as proxy

HP_MIN, HP_MAX = 60, 300

p_min = df['Present_Price'].min()
p_max = df['Present_Price'].max()

df['Horsepower'] = (
    (df['Present_Price'] - p_min) / (p_max - p_min)  
    * (HP_MAX - HP_MIN)                                
    + HP_MIN                                           
).round(1)

print(df[['Car_Name', 'Present_Price', 'Horsepower']].head(10))

In [ ]:
df.head()

In [ ]:
#Store above features in df1

df1 = df[['Mileage_per_year','Horsepower','Brand_Goodwill_Score','Selling_Price']].copy()
df1.head()

In [ ]:
df1.shape

<h2>Data Cleaning</h2>

In [ ]:
#Checking for null values

df1.isnull().sum()

In [ ]:
#Checking for duplicate values

df1.duplicated().sum()

In [ ]:
#Remove Duplicates

df1 = df1.drop_duplicates().copy()

In [ ]:
df1.describe()

<h2>Data Preprocessing</h2>

In [ ]:
#Normalise Features using Min Max Scaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
FEATURES = ['Mileage_per_year', 'Horsepower', 'Brand_Goodwill_Score']
 
X_raw = df1[FEATURES]
y     = df1['Selling_Price']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Then fit scaler ONLY on train, transform both
scaler  = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=FEATURES)
X_test  = pd.DataFrame(scaler.transform(X_test_raw),      columns=FEATURES)

<h2>Visualising Data</h2>

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib inline

def scatter_plot(X_train,y):
    plt.scatter(X_train['Mileage_per_year'],y,marker='+',label='Mileage')
    plt.scatter(X_train['Horsepower'],y,label='Horsepower')
    plt.scatter(X_train['Brand_Goodwill_Score'],y,marker='o',label='Goodwill')
    plt.xlabel('Car Features')
    plt.ylabel('Car Present Price')
    plt.legend()
    plt.show()

scatter_plot(X_train,y_train)

<h2>Remove Outliers using Z score</h2>

In [ ]:
def zscore_mask(series, threshold=3):
    """Flag rows where |z| > threshold as outliers."""
    z_scores = (series - series.mean()) / series.std()
    return z_scores.abs() <= threshold

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

mask_z = pd.Series([True] * len(X_train), index=X_train.index)
for col in FEATURES:
    mask_z = mask_z & zscore_mask(X_train[col]) 

In [ ]:
X_clean = X_train[mask_z]
y_clean = y_train[mask_z]

<h2>Clean Data Visualisation</h2>

In [ ]:
scatter_plot(X_clean,y_clean)

<h2>Model Building</h2>

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [ ]:
cross_val_score(LinearRegression(),X_clean,y_clean,cv=5)

In [ ]:
#Training the model using train data

model = LinearRegression()
model.fit(X_clean,y_clean)

In [ ]:
#Making predictions

y_pred = model.predict(X_test)

In [ ]:
#Calculating accuracy of model
score = model.score(X_test, y_test)

print(f"Accuracy of model = {score}")

<h2>Model Evaluation</h2>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error = {mae}")
print(f"Mean Squared Error = {mse}")
print(f"R-squared Error = {r2:.2f}")

<h2>Model Visualisation</h2>

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib inline

plt.scatter(y_test, y_pred)
plt.xlabel('Actual points')
plt.ylabel('Predicted Points')
plt.show()

<h2>Saving Model</h2>

In [ ]:
import joblib
joblib.dump(model,"Car Price Prediction.pkl")